# QData Free Source Factor API Arithmetic

This notebook is a concise walkthrough of deterministic factor API timing arithmetic on the QData mock backend. It does not require Docker, paid data, pandas, or external network access.

The demo treats `momentum_20d` as an after-close signal on `2024-01-02`, ranks the synthetic HS300 mock universe, fills at the next session open, and marks at that session close. This is fixture arithmetic, not strategy performance or real-market evidence.

In [1]:
from pathlib import Path
import sys

repo_root = Path.cwd()
if not (repo_root / "qdata").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

from qdata import Client
from examples.factor_api_arithmetic_demo import run_demo, format_report

## 1. Build a tradable universe

A quant research database should not blindly rank every listed security. The first step is to apply tradability filters such as suspension, ST status, delisting period and minimum listing days.

In [2]:
client = Client(default_format="records")
tradable = client.get_tradable_universe(
    asof_date="2024-01-02",
    universe="hs300",
    min_list_days=120,
)
tradable

[{'symbol': '600519.SH',
  'security_id': 1000001,
  'asof_date': '2024-01-02',
  'can_buy': True,
  'can_sell': True,
  'list_days': 8163,
  'is_st': False,
  'is_suspended': False,
  'is_new_listing': False,
  'is_delisting_period': False},
 {'symbol': '000001.SZ',
  'security_id': 1000002,
  'asof_date': '2024-01-02',
  'can_buy': True,
  'can_sell': True,
  'list_days': 11962,
  'is_st': False,
  'is_suspended': False,
  'is_new_listing': False,
  'is_delisting_period': False}]

## 2. Pull point-in-time factor values

The factor signal is requested as of the signal date. In a real setup, this prevents future data leakage.

In [3]:
symbols = [row["symbol"] for row in tradable]
client.get_factor(
    factors=["momentum_20d", "roe_ttm"],
    symbols=symbols,
    start_date="2024-01-02",
    end_date="2024-01-02",
    format="wide",
)

[{'symbol': '600519.SH',
  'security_id': 1000001,
  'trade_date': '2024-01-02',
  'momentum_20d': 0.032,
  'roe_ttm': 0.283},
 {'symbol': '000001.SZ',
  'security_id': 1000002,
  'trade_date': '2024-01-02',
  'momentum_20d': -0.011,
  'roe_ttm': 0.104}]

## 3. Run the timing arithmetic

The script version lives in `examples/factor_api_arithmetic_demo.py`. It uses only the Python standard library plus the QData SDK.

In [4]:
result = run_demo()
print(format_report(result))

QData factor API arithmetic demo
universe=hs300 factor=momentum_20d signal_date=2024-01-02 execution_date=2024-01-03
signal_timing=after_close fill_timing=next_session_open mark_timing=next_session_close
tradable_symbols=2 long_bucket=600519.SH short_bucket=000001.SZ
long_return=0.5307% benchmark_return=0.7883% active_return=-0.2577% factor_spread=-0.5154%


## 4. What this demonstrates

- The research workflow starts from a tradable universe instead of raw symbols.
- Factor values are pulled by signal date to avoid future leakage.
- Prices are forward-adjusted through the SDK.
- The result reports deterministic bucket and benchmark arithmetic from the synthetic fixture.

The dataset is intentionally tiny so the notebook is easy to review as a technical walkthrough. These numbers check API and timing alignment only; they are not strategy performance, real-market evidence, or investment advice. Real database behavior requires separate integration verification.